# WavLM-Base+ Utterance Embedding Extraction

## Goal: Extract 768-dim WavLM embeddings for 15,060 utterances

This is Step 1 of the fusion pipeline. WavLM captures learned audio
representations that should dramatically outperform hand-crafted features.

Previous result: librosa F0/RMS/ZCR/MFCC → F1 = 0.29 ceiling
Expected: WavLM audio-only → F1 = 0.55-0.65 at utterance level

## Setup
1. Add shared GDrive folder as shortcut
2. Run all cells (~2-3 hours on T4 GPU)

In [ ]:
!pip install transformers librosa scipy scikit-learn -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, time
import numpy as np
from pathlib import Path
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import librosa
from functools import lru_cache

# Find data
DATA_PATH = None
for p in ['/content/drive/MyDrive/laughter_prediction_backup',
          '/content/drive/MyDrive/My Drive/laughter_prediction_backup']:
    if os.path.exists(p):
        DATA_PATH = p
        break

if not DATA_PATH:
    import subprocess
    result = subprocess.run(['find', '/content/drive/MyDrive', '-name', 'aligned_utterances.jsonl', '-maxdepth', '4'],
                          capture_output=True, text=True, timeout=30)
    if result.stdout.strip():
        found = result.stdout.strip().split('\n')[0]
        DATA_PATH = os.path.dirname(os.path.dirname(found))

if DATA_PATH:
    print(f'✅ Data found: {DATA_PATH}')
else:
    print('❌ Data not found! Add GDrive shortcut:')
    print('1. Open: https://drive.google.com/drive/folders/15ixKiy86MZ67OwGEVxtnwSTs3nvbLRbh')
    print('2. Click "Add shortcut to Drive"')

In [ ]:
# Load utterances
utterances = []
utt_path = os.path.join(DATA_PATH, 'aligned_utterances.jsonl') if DATA_PATH else None

if utt_path and os.path.exists(utt_path):
    with open(utt_path) as f:
        for line in f:
            try:
                utterances.append(json.loads(line))
            except:
                pass
    print(f'✅ Loaded {len(utterances):,} utterances')
    pos = sum(1 for u in utterances if u.get('label_any') == 1)
    print(f'   Positive: {pos:,} ({pos/len(utterances)*100:.1f}%)')
    print(f'   Videos: {len(set(u["video_id"] for u in utterances))}')
else:
    # Fallback: create utterances from word-level segments
    print('Creating utterances from word-level segments...')
    segs_path = os.path.join(DATA_PATH, 'aligned_segments.jsonl')
    from collections import defaultdict
    video_segs = defaultdict(list)
    with open(segs_path) as f:
        for line in f:
            try:
                seg = json.loads(line)
                if 'start' in seg:
                    video_segs[seg['video_id']].append(seg)
            except:
                pass
    
    for vid, segs in video_segs.items():
        segs.sort(key=lambda x: x['start'])
        # Group into utterances of ~25 words
        for i in range(0, len(segs), 25):
            chunk = segs[i:i+25]
            text = ' '.join(s.get('word', '') for s in chunk)
            label = 1 if any(s.get('label', 0) == 1 for s in chunk) else 0
            utterances.append({
                'utterance_id': f'{vid}_{i//25:04d}',
                'video_id': vid,
                'text': text,
                'start': chunk[0]['start'],
                'end': chunk[-1]['end'],
                'duration': chunk[-1]['end'] - chunk[0]['start'],
                'n_words': len(chunk),
                'label_any': label,
                'audio_file': chunk[0].get('audio_file', ''),
            })
    print(f'✅ Created {len(utterances):,} utterances from segments')

# Find audio files
audio_path = os.path.join(DATA_PATH, 'audio') if DATA_PATH else None
audio_on_disk = {}
if audio_path and os.path.exists(audio_path):
    for root, dirs, files in os.walk(audio_path):
        for f in files:
            if f.endswith('.mp3'):
                vid = f.replace('.mp3', '')
                audio_on_disk[vid] = os.path.join(root, f)
    print(f'Audio files: {len(audio_on_disk)}')

In [ ]:
# Load WavLM model
print('Loading WavLM-Base+...')
model_name = 'microsoft/wavlm-base-plus'
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
wavlm = WavLMModel.from_pretrained(model_name).to(DEVICE)
wavlm.eval()
print(f'✅ WavLM loaded on {DEVICE}')
print(f'   Parameters: {sum(p.numel() for p in wavlm.parameters()):,}')

SR = 16000
MAX_DURATION = 30.0  # Max utterance duration in seconds
EMBEDDING_DIM = 768

In [ ]:
# Extract WavLM embeddings for all utterances
@lru_cache(maxsize=20)
def load_audio_cached(audio_path):
    """Cache audio files to avoid re-loading."""
    try:
        y, sr = librosa.load(audio_path, sr=SR, duration=600)  # First 10 min
        return y
    except:
        return None

def extract_wavlm_embedding(y_segment):
    """Extract WavLM embedding from audio segment."""
    if y_segment is None or len(y_segment) < SR * 0.1:
        return np.zeros(EMBEDDING_DIM)
    
    # Prepare input
    inputs = feature_extractor(
        y_segment, sampling_rate=SR, return_tensors='pt', padding=True
    )
    
    with torch.no_grad():
        outputs = wavlm(input_values=inputs.input_values.to(DEVICE))
    
    # Mean pool over time dimension
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding

# Process utterances by video (load audio once per video)
embeddings = {}
failed = 0
t0 = time.time()

video_ids = sorted(set(u['video_id'] for u in utterances))
print(f'Processing {len(utterances):,} utterances from {len(video_ids)} videos...')

for vi, vid in enumerate(video_ids):
    vid_utts = [u for u in utterances if u['video_id'] == vid]
    
    # Find audio
    audio_file = vid_utts[0].get('audio_file', '')
    if not audio_file or not os.path.exists(audio_file):
        if vid in audio_on_disk:
            audio_file = audio_on_disk[vid]
        else:
            # No audio for this video
            for u in vid_utts:
                embeddings[u['utterance_id']] = np.zeros(EMBEDDING_DIM)
                failed += 1
            continue
    
    # Load full audio (cached)
    y = load_audio_cached(audio_file)
    if y is None:
        for u in vid_utts:
            embeddings[u['utterance_id']] = np.zeros(EMBEDDING_DIM)
            failed += 1
        continue
    
    # Extract per-utterance embedding
    for u in vid_utts:
        start = u['start']
        end = min(u['end'], u['start'] + MAX_DURATION)
        
        # Add 200ms context padding
        s = max(0, start - 0.2)
        e = min(len(y) / SR, end + 0.2)
        
        segment = y[int(s * SR):int(e * SR)]
        if len(segment) < SR * 0.1:  # Too short
            embeddings[u['utterance_id']] = np.zeros(EMBEDDING_DIM)
            failed += 1
            continue
        
        emb = extract_wavlm_embedding(segment)
        embeddings[u['utterance_id']] = emb
    
    # Clear cache periodically to manage memory
    if vi % 10 == 0:
        load_audio_cached.cache_clear()
    
    if (vi + 1) % 5 == 0:
        elapsed = time.time() - t0
        print(f'  [{vi+1}/{len(video_ids)}] {len(embeddings):,} embeddings ({elapsed:.0f}s)')

total_time = time.time() - t0
print(f'\n✅ Extracted {len(embeddings):,} embeddings in {total_time:.0f}s')
print(f'   Failed: {failed} ({failed/len(embeddings)*100:.1f}%)')
print(f'   Embedding dim: {EMBEDDING_DIM}')

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

# Quick evaluation: WavLM audio-only utterance classification
print('=' * 70)
print('WAVLM AUDIO-ONLY: UTTERANCE-LEVEL CLASSIFICATION')
print('=' * 70)

# Build feature matrix
X_list = []
y_list = []
vid_list = []
missing = 0

for u in utterances:
    uid = u['utterance_id']
    if uid in embeddings:
        emb = embeddings[uid]
        if np.all(emb == 0):
            missing += 1
            continue
        X_list.append(emb)
        y_list.append(u.get('label_any', 0))
        vid_list.append(u['video_id'])
    else:
        missing += 1

X = np.array(X_list)
y = np.array(y_list)
videos = np.array(vid_list)

print(f'Dataset: {len(X):,} utterances')
print(f'Positive: {y.sum():,} ({y.mean()*100:.1f}%)')
print(f'Negative: {(1-y).sum():,}')
print(f'Missing/zero embeddings: {missing}')
print(f'Embedding dim: {X.shape[1]}')

# Per-video split
unique_vids = np.unique(videos)
np.random.seed(42)
np.random.shuffle(unique_vids)
n80 = int(0.8 * len(unique_vids))
train_vids = set(unique_vids[:n80])
test_vids = set(unique_vids[n80:])

train_idx = np.array([i for i, v in enumerate(videos) if v in train_vids])
test_idx = np.array([i for i, v in enumerate(videos) if v in test_vids])

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f'\nTrain: {len(X_train):,} from {len(train_vids)} videos')
print(f'Test:  {len(X_test):,} from {len(test_vids)} videos')

# Logistic Regression
clf = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)
lr_f1 = f1_score(y_test, clf.predict(X_test))
print(f'\nLogistic Regression: F1 = {lr_f1:.4f}')

# Random Forest
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
rf_f1 = f1_score(y_test, rf.predict(X_test))
print(f'Random Forest:       F1 = {rf_f1:.4f}')

# Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
gb_f1 = f1_score(y_test, gb.predict(X_test))
print(f'Gradient Boosting:   F1 = {gb_f1:.4f}')

print(f'\nBaseline comparison:')
print(f'  Pause only (word-level):      F1 = 0.20')
print(f'  All acoustic features (word):  F1 = 0.29')
print(f'  TF-IDF text (word-level):     F1 = 0.73')
print(f'  XLM-R text (word-level):       F1 = 0.82')
print(f'  WavLM audio (utterance):       F1 = {lr_f1:.4f}')

if lr_f1 > 0.50:
    print(f'\n✅ WavLM BREAKS THE 0.30 CEILING! Audio is useful at utterance level.')
elif lr_f1 > 0.40:
    print(f'\n⚠️ PARTIAL: WavLM improves over librosa but below target.')
else:
    print(f'\n❌ WavLM still below target. Consider fusion with text.')

In [ ]:
# Save embeddings for later fusion
import torch

# Convert to tensors
emb_dict = {uid: torch.tensor(emb, dtype=torch.float32) for uid, emb in embeddings.items()}

save_path = os.path.join(DATA_PATH if DATA_PATH else '.', 'wavlm_utterance_embeddings.pt')
torch.save(emb_dict, save_path)
print(f'✅ Saved {len(emb_dict)} embeddings to {save_path}')
print(f'   File size: {os.path.getsize(save_path) / 1e6:.1f} MB')

# Also save as numpy for sklearn
np.save(os.path.join(DATA_PATH if DATA_PATH else '.', 'wavlm_embeddings.npy'), X)
np.save(os.path.join(DATA_PATH if DATA_PATH else '.', 'wavlm_labels.npy'), y)
np.save(os.path.join(DATA_PATH if DATA_PATH else '.', 'wavlm_video_ids.npy'), videos)
print(f'✅ Saved numpy arrays')

# Classification report
print('\n' + '=' * 70)
print('CLASSIFICATION REPORT')
print('=' * 70)
print(classification_report(y_test, clf.predict(X_test), target_names=['non-laugh', 'laugh']))